# AirbagsCV — Colab Training Notebook

This notebook trains the EfficientAD anomaly-detection model on the AITEX
dataset using **the repository's own scripts**. It does NOT reimplement
training logic. The repo is the source of truth.

**Before running:**
1. Upload AITEX to Google Drive at `/MyDrive/datasets/AITEX/` (must contain
   `NODefect_images/`, `Defect_images/`, `Mask_images/`).
2. Upload Imagenette (extracted from `imagenette2-160.tgz` at
   https://github.com/fastai/imagenette) to `/MyDrive/datasets/imagenette/`.
3. Select GPU runtime: `Runtime > Change runtime type > Hardware
   accelerator > GPU`.

See `notebooks/COLAB_GUIDE.md` in the repo for the full guide and
troubleshooting table.

## 1. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Runtime > Change runtime type > GPU, then Runtime > Restart session.')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
aitex_path = '/content/drive/MyDrive/datasets/AITEX'
if not os.path.isdir(aitex_path):
    raise FileNotFoundError(
        f'{aitex_path} does not exist. Upload AITEX first (NODefect_images/, '
        f'Defect_images/, Mask_images/).'
    )
print('AITEX contents:', os.listdir(aitex_path))

## 3. Clone or update the repo

In [ ]:
%%bash
if [ -d /content/AirbagsCV ]; then
  cd /content/AirbagsCV
  git pull
else
  cd /content
  git clone https://github.com/MohamedAzzam4/AirbagsCV.git
fi

In [ ]:
%cd /content/AirbagsCV

## 4. Install dependencies

Colab ships with `torch` + `torchvision` preinstalled (CUDA build). Do NOT
reinstall torch from the default index — you will lose GPU.

In [ ]:
%%bash
cd /content/AirbagsCV
pip install -r requirements-colab.txt 2>&1 | tail -20

## 5. Define paths (default Drive-persistent paths)

All outputs land on Drive so they survive Colab disconnects.

In [ ]:
import os
os.environ['REPO_DIR']          = '/content/AirbagsCV'
os.environ['RAW_AITEX_DIR']     = '/content/drive/MyDrive/datasets/AITEX'
os.environ['IMAGENETTE_DIR']    = '/content/drive/MyDrive/datasets/imagenette/train'
os.environ['PREPARED_DATA_DIR'] = '/content/drive/MyDrive/AirbagsCV/prepared/aitex_patches'
os.environ['RUNS_DIR']          = '/content/drive/MyDrive/AirbagsCV/runs'
os.environ['RESULTS_DIR']       = '/content/drive/MyDrive/AirbagsCV/results'
os.environ['CACHE_DIR']         = '/content/drive/MyDrive/AirbagsCV/cache'

for d in ['PREPARED_DATA_DIR', 'RUNS_DIR', 'RESULTS_DIR', 'CACHE_DIR']:
    os.makedirs(os.environ[d], exist_ok=True)

for k, v in os.environ.items():
    if k in ['REPO_DIR','RAW_AITEX_DIR','IMAGENETTE_DIR','PREPARED_DATA_DIR',
             'RUNS_DIR','RESULTS_DIR','CACHE_DIR']:
        print(f'{k:25s} {v}')

In [ ]:
# Verify datasets are reachable
required = ['NODefect_images', 'Defect_images', 'Mask_images']
for r in required:
    p = os.path.join(os.environ['RAW_AITEX_DIR'], r)
    if not os.path.isdir(p):
        raise FileNotFoundError(f'Missing {p}. Check RAW_AITEX_DIR or upload AITEX to Drive.')
print('AITEX OK')

if not os.path.isdir(os.environ['IMAGENETTE_DIR']):
    raise FileNotFoundError(
        f"Missing imagenette at {os.environ['IMAGENETTE_DIR']}. "
        f"Download imagenette2-160.tgz from https://github.com/fastai/imagenette "
        f"and extract to /content/drive/MyDrive/datasets/imagenette/."
    )
subdirs = [d for d in os.listdir(os.environ['IMAGENETTE_DIR'])
           if os.path.isdir(os.path.join(os.environ['IMAGENETTE_DIR'], d))]
if not subdirs:
    raise FileNotFoundError(
        f"{os.environ['IMAGENETTE_DIR']} exists but has no class subdirs. "
        f"It must be ImageFolder layout: imagenette/train/<class>/*.JPEG"
    )
print(f'Imagenette OK ({len(subdirs)} classes)')

## 6. Smoke test

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/smoke_test.py

## 7. Prepare AITEX data

Writes the Anomalib `Folder` layout to `PREPARED_DATA_DIR` (on Drive by default).

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/prepare_aitex.py \
  --source "$RAW_AITEX_DIR" \
  --output "$PREPARED_DATA_DIR" \
  --patch-size 256 \
  --blank-threshold 0.5 \
  --seed 42 \
  --overwrite

## 8. Choose Mode A or Mode B (Drive I/O strategy)

**Mode A (default, safer):** train directly from `PREPARED_DATA_DIR` on
Drive. Safer after disconnect. Slower (Drive I/O per batch). To use Mode A,
do nothing — skip to step 9.

**Mode B (faster):** copy prepared data from Drive to `/content` at the
start of the runtime, train from there, save checkpoints back to Drive.
Faster (local SSD I/O). Run the cell below to enable Mode B.

In [ ]:
# === OPTIONAL: Mode B — copy prepared data from Drive to /content for faster I/O ===
# Skip this cell to use Mode A (train directly from Drive).
import os, shutil
LOCAL_DATA_DIR = '/content/aitex_patches_local'
if os.path.exists(LOCAL_DATA_DIR):
    print(f'{LOCAL_DATA_DIR} already exists; reusing.')
else:
    print(f'Copying {os.environ["PREPARED_DATA_DIR"]} -> {LOCAL_DATA_DIR} ...')
    shutil.copytree(os.environ['PREPARED_DATA_DIR'], LOCAL_DATA_DIR)
    print('Done.')
os.environ['PREPARED_DATA_DIR_DRIVE'] = os.environ['PREPARED_DATA_DIR']
os.environ['PREPARED_DATA_DIR'] = LOCAL_DATA_DIR
print('PREPARED_DATA_DIR is now:', os.environ['PREPARED_DATA_DIR'])
print('  (checkpoints/results still go to RUNS_DIR/RESULTS_DIR on Drive)')

## 9. Train EfficientAD (from scratch)

EfficientAD **requires `--batch-size 1`** (Anomalib 2.0.0 hard constraint;
the script warns and overrides any other value). EfficientAD **also
requires `--imagenet-dir`** pointing at imagenette.

70 epochs on a T4 GPU takes ~30-60 minutes. The script prints all key
paths at the end (RUN_DIR, LAST_CHECKPOINT_PATH, BEST_CHECKPOINT_PATH, etc.).

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/train_models.py \
  --model efficientad \
  --dataset aitex \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-dir "$RUNS_DIR" \
  --epochs 70 \
  --batch-size 1 \
  --num-workers 2 \
  --accelerator gpu \
  --devices 1 \
  --precision 16-mixed \
  --seed 42 \
  --imagenet-dir "$IMAGENETTE_DIR"

## 9b. Train PatchCore (alternative model, no imagenette needed)

PatchCore is a memory-bank anomaly-detection model. Unlike EfficientAD:
- **No `--imagenet-dir` required.**
- **Larger batch size OK** (default 8, vs EfficientAD's forced 1).
- **Only 1 epoch needed** — the memory bank is fitted once.
- **No optimizer state** — 'resume' is conceptually meaningless. To evaluate, use `benchmark_inference.py` or `demo/inference.py`.

Skip this cell if you only want EfficientAD.

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/train_models.py \
  --model patchcore \
  --dataset aitex \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-dir "$RUNS_DIR" \
  --epochs 1 \
  --batch-size 8 \
  --num-workers 2 \
  --accelerator gpu \
  --devices 1 \
  --precision 16-mixed \
  --seed 42 \
  --backbone wide_resnet50_2 \
  --coreset-sampling-ratio 0.1 \
  --save-top-k 1

Outputs land in `$RUNS_DIR/patchcore_aitex/` (same layout as EfficientAD).
To evaluate: use `scripts/benchmark_inference.py --model patchcore --checkpoint <path>`
or `demo/app.py` with the PatchCore dropdown option. Do NOT pass
`--resume-from-checkpoint` to PatchCore (it would re-run fit() and rebuild
the memory bank from scratch).

## 10. Robust resume (after disconnect)

This cell remounts Drive, pulls latest repo, reinstalls deps if needed,
redefines env vars, finds the latest `last.ckpt`, and resumes training.
If no checkpoint exists, it tells you to run step 9 (train from scratch).

In [ ]:
# === ROBUST RESUME CELL ===
# 1. Remount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Pull latest repo (clone if missing)
import subprocess, os
repo_dir = '/content/AirbagsCV'
if not os.path.isdir(repo_dir):
    subprocess.run(['git', 'clone', 'https://github.com/MohamedAzzam4/AirbagsCV.git', repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

# 3. Check if dependencies are installed; reinstall if not
try:
    import anomalib  # noqa: F401
    print('Dependencies OK.')
except ImportError:
    print('anomalib missing; reinstalling dependencies...')
    subprocess.run(['pip', 'install', '-r', f'{repo_dir}/requirements-colab.txt'], check=True)

# 4. Redefine env vars (must match step 5)
os.environ['REPO_DIR']          = '/content/AirbagsCV'
os.environ['RAW_AITEX_DIR']     = '/content/drive/MyDrive/datasets/AITEX'
os.environ['IMAGENETTE_DIR']    = '/content/drive/MyDrive/datasets/imagenette/train'
os.environ['PREPARED_DATA_DIR'] = '/content/drive/MyDrive/AirbagsCV/prepared/aitex_patches'
os.environ['RUNS_DIR']          = '/content/drive/MyDrive/AirbagsCV/runs'
os.environ['RESULTS_DIR']       = '/content/drive/MyDrive/AirbagsCV/results'
os.environ['CACHE_DIR']         = '/content/drive/MyDrive/AirbagsCV/cache'
for d in ['PREPARED_DATA_DIR', 'RUNS_DIR', 'RESULTS_DIR', 'CACHE_DIR']:
    os.makedirs(os.environ[d], exist_ok=True)

# 5. Find latest checkpoint under RUNS_DIR (prefers last.ckpt)
import glob
run_dir = os.path.join(os.environ['RUNS_DIR'], 'efficientad_aitex')
last_ckpts = sorted(
    glob.glob(f'{run_dir}/**/last.ckpt', recursive=True),
    key=lambda p: os.path.getmtime(p),
)
all_ckpts = sorted(
    glob.glob(f'{run_dir}/**/*.ckpt', recursive=True),
    key=lambda p: os.path.getmtime(p),
)
if last_ckpts:
    CKPT = last_ckpts[-1]
    print(f'Resuming from last.ckpt: {CKPT}')
elif all_ckpts:
    CKPT = all_ckpts[-1]
    print(f'WARNING: no last.ckpt found; resuming from newest checkpoint: {CKPT}')
else:
    CKPT = None
    print(f'NO CHECKPOINT FOUND under {run_dir}.')
    print('Training must start from zero — run step 9 (Train EfficientAD from scratch).')

In [ ]:
# Run this cell ONLY if CKPT was set above.
# (If CKPT is None, skip this cell and run step 9 instead.)
%%bash -s "$CKPT"
cd /content/AirbagsCV
python scripts/train_models.py \
  --model efficientad \
  --dataset aitex \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-dir "$RUNS_DIR" \
  --epochs 70 \
  --batch-size 1 \
  --num-workers 2 \
  --accelerator gpu \
  --devices 1 \
  --precision 16-mixed \
  --seed 42 \
  --imagenet-dir "$IMAGENETTE_DIR" \
  --resume-from-checkpoint "$1"

## 11. Run honest inference benchmark

Measures real per-image latency with warmup + CUDA sync. Reports p50/p95/p99.

In [ ]:
# Set CHECKPOINT_PATH — prefer last.ckpt, then model.ckpt, then any, then repo's committed ckpt
import glob, os
run_dir = os.path.join(os.environ['RUNS_DIR'], 'efficientad_aitex')
candidates = []
for pattern in ['last.ckpt', 'model.ckpt']:
    found = sorted(glob.glob(f'{run_dir}/**/{pattern}', recursive=True),
                   key=lambda p: os.path.getmtime(p))
    candidates.extend(found)
candidates.extend(sorted(glob.glob(f'{run_dir}/**/*.ckpt', recursive=True),
                         key=lambda p: os.path.getmtime(p)))
if not candidates:
    candidates = ['results/efficientad_aitex/EfficientAd/aitex/latest/weights/lightning/model.ckpt']
os.environ['CHECKPOINT_PATH'] = candidates[0]
print('CHECKPOINT_PATH =', os.environ['CHECKPOINT_PATH'])

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/benchmark_inference.py \
  --checkpoint "$CHECKPOINT_PATH" \
  --model efficientad \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-csv "$RESULTS_DIR/latency.csv" \
  --device cuda \
  --batch-size 1 \
  --warmup 20 \
  --iterations 200

## 12. Verify outputs are on Drive

In [ ]:
import os
for label, path in [('Runs dir', os.environ['RUNS_DIR']),
                    ('Results dir', os.environ['RESULTS_DIR'])]:
    print(f'\n=== {label}: {path} ===')
    n = 0
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
            n += 1
        if len(files) > 5:
            print(f"{indent}  ... and {len(files)-5} more")
    print(f'Total visible files: {n}')

## 13. Zip outputs (optional)

In [ ]:
%%bash
cd /content
zip -r /content/drive/MyDrive/AirbagsCV/airbagcv_run_outputs.zip \
  "$RUNS_DIR" \
  "$RESULTS_DIR" 2>&1 | tail -5
ls -lh /content/drive/MyDrive/AirbagsCV/airbagcv_run_outputs.zip

## Done

Your trained checkpoint, metrics, and benchmark are on Google Drive at
`/MyDrive/AirbagsCV/`. To run the interactive demo locally:

1. Download your checkpoint from Drive (e.g. `last.ckpt` or `model.ckpt`).
2. Clone the repo on your local machine.
3. Place the checkpoint under
   `results/efficientad_aitex/EfficientAd/aitex/latest/weights/lightning/model.ckpt`.
4. Run `python demo/app.py` and open http://127.0.0.1:7860/.

See `notebooks/COLAB_GUIDE.md` for the troubleshooting table.